In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
spark.version

'4.1.1'

In [8]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [9]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df.registerTempTable('trips_data')

C:\Users\dfrei\data_projects\spark\.venv\Lib\site-packages\pyspark\sql\classic\dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [13]:
spark.sql("""
SELECT 
    count(1) as total_trips
FROM 
    trips_data
WHERE
    MONTH(tpep_pickup_datetime) = 11 AND DAY(tpep_pickup_datetime) = 15;

""").show()

+-----------+
|total_trips|
+-----------+
|     162604|
+-----------+



In [24]:
spark.sql("""
SELECT 
    (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600 AS longest_trip
FROM 
    trips_data
ORDER BY
    longest_trip DESC
LIMIT 1;
""").show()

+-----------------+
|     longest_trip|
+-----------------+
|90.64666666666666|
+-----------------+



In [27]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [28]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [29]:
df_result = df.join(df_zones,df.PULocationID == df_zones.LocationID)

In [30]:
df_result.registerTempTable('joined_data')

In [38]:
spark.sql("""
SELECT 
    Zone,
    COUNT(1) AS total_trips
FROM 
    joined_data
GROUP BY
    Zone
ORDER BY
    total_trips
LIMIT 5
;
""").show()

+--------------------+-----------+
|                Zone|total_trips|
+--------------------+-----------+
|Governor's Island...|          1|
|Eltingville/Annad...|          1|
|       Arden Heights|          1|
|       Port Richmond|          3|
|       Rikers Island|          4|
+--------------------+-----------+

